In [1]:
import torch
import torch.nn as nn 
import random, math
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import matplotlib

In [2]:
IMAGE_SIZE=32
EMDED_DIM=64
ATTENTION_HEAD=4
BATCH_SIZE=12
EPOCHS=10
LR=3e-10
TEMP=0.07

In [3]:
device=torch.device('mps' if torch.mps.is_available() else 'cpu')

In [4]:

colors = ['red', 'green', 'blue', 'yellow', 'purple', 'orange', 'pink', 'brown', 'gray']
shapes = ['square', 'circle', 'triangle']
positions = ['left', 'center', 'right', 'top', 'bottom', 'top-left', 'top-right', 'bottom-left', 'bottom-right']


In [5]:

from PIL import Image, ImageDraw

IMG_SIZE = 64

def draw_sample(color, shape, position, img_size=IMG_SIZE):
    img = Image.new('RGB', (img_size, img_size), 'white')
    draw = ImageDraw.Draw(img)

    margin = 6
    h = w = img_size - 2 * margin

    # Calculate x coordinates
    if 'left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'top-left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'bottom-left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    elif 'top-right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    elif 'bottom-right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    else:  # center or vertical positions
        x0 = margin + w // 4
        x1 = margin + 3 * w // 4

    # Calculate y coordinates
    if 'top' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'top-left' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'top-right' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'bottom' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    elif 'bottom-left' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    elif 'bottom-right' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    else:  # center or horizontal positions
        y0 = margin + h // 4
        y1 = margin + 3 * h // 4

    if shape == 'square':
        draw.rectangle([x0, y0, x1, y1],
                       fill=color, outline='black')
    elif shape == 'circle':
        draw.ellipse([x0, y0, x1, y1],
                     fill=color, outline='black')
    else:  # triangle
        draw.polygon(
            [(x0 + (x1 - x0) // 2, y0),
             (x0, y1),
             (x1, y1)],
            fill=color,
            outline='black'
        )

    return img


In [6]:
#building the dataset 
import numpy as np

class shapedataset():
    def __init__(self):
        self.images=[]
        self.captions=[]

        for c in colors:
            for s in shapes:
                for p in positions:
                    img=draw_sample(c,s,p)
                    cap=f"{c} {s} {p}"

                    self.images.append(torch.from_numpy(np.array(img, copy=True)).permute(2, 0, 1).float() / 255.0)
                    self.captions.append(cap)

        self.vocab, self.word2idx=self.build_vocab(self.captions)

    def build_vocab(self,texts):
        words= sorted({w for t in texts for w in t.split()})
        vocab= ['[CLS]']+ words
        w2i={w:i for i,w in enumerate(vocab)}
        return vocab,w2i
    def encode_text(self,text):
        toks = [self.word2idx['[CLS]']] + [self.word2idx[w] for w in text.split()]
        return torch.tensor(toks, dtype=torch.long)
    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
         return self.images[i], self.encode_text(self.captions[i]), self.captions[i]


full_ds = shapedataset()
VOCAB_SIZE = len(full_ds.vocab)

In [7]:
print(VOCAB_SIZE)
print(full_ds.vocab)

22
['[CLS]', 'blue', 'bottom', 'bottom-left', 'bottom-right', 'brown', 'center', 'circle', 'gray', 'green', 'left', 'orange', 'pink', 'purple', 'red', 'right', 'square', 'top', 'top-left', 'top-right', 'triangle', 'yellow']


In [9]:
# Train-val dataset

train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size

train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [train_size, val_size]
)

In [10]:
train_loader= DataLoader(train_ds, batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [11]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.convolutions=nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32,
                      kernel_size=3, stride=2, padding=1 ),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64,
                              kernel_size=3, stride=2, padding=1 ),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128,
                                  kernel_size=3, stride=2, padding=1 ), 
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=256,
                                  kernel_size=3, stride=2, padding=1 ) ,
            nn.ReLU(), 

        )

        self.projection=nn.Linear(256, 64)
        self.layernorm1=nn.LayerNorm(64)
    def forward(self,x):
        x= self.convolutions(x)
        x= x.mean(dim=[2,3]) #along height, width
        x=self.projection(x)
        x=F.normalize(self.layernorm1(x), dim=-1)
        return x



In [12]:
class TextEncoder(nn.Module):
    def __init__(self, embed_dim=EMDED_DIM, num_heads=ATTENTION_HEAD, vocab_size=VOCAB_SIZE, context_size=4):
        super().__init__()

        self.token_embedding=nn.Embedding(vocab_size, embed_dim)
        self.postion_embedding=nn.Embedding(vocab_size, context_size)
        self.mha= nn.MultiheadAttention(embed_dim, num_heads,)
        self.norm=nn.LayerNorm(embed_dim)

    def forward(self,x):
        pass